# CLASP and IMSTE across all single-cell datasets

This notebook processes every original H5AD dataset found in `data/`, selecting a configurable number of highly variable genes (2,000 by default), scaling them, and reducing them to a configurable number of PCs (50 by default), using up to 4,000 cells per dataset. For each dataset it fits the CLASP graph, UMAP and IMSTE embeddings, then plots batch-provenance and label views plus the kNN neighborhood union for both embeddings. Choose Matplotlib or Datashader for the embedding views in the controls below. The dataset loop runs in order when the final analysis cell is executed.

In [6]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

from pathlib import Path
import importlib.util
import inspect
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "src" / "clasp").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find the CLASP project root from the notebook working directory")
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
for import_path in (REPO_ROOT / "src", NOTEBOOK_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

import imste_single_cell
import clasp_imste_workflow as workflow

IMSTE_EMBEDDER_CLASS = imste_single_cell.load_embedder_class()
IMSTE_VISUALIZATION_PATH = Path(inspect.getfile(IMSTE_EMBEDDER_CLASS)).parent / "visualization.py"
if not IMSTE_VISUALIZATION_PATH.is_file():
    raise FileNotFoundError(f"IMSTE visualization utilities not found: {IMSTE_VISUALIZATION_PATH}")
_visualization_spec = importlib.util.spec_from_file_location(
    "imste_visualization_tools", IMSTE_VISUALIZATION_PATH
)
imste_visualization = importlib.util.module_from_spec(_visualization_spec)
_visualization_spec.loader.exec_module(imste_visualization)
plot_embedding = imste_single_cell.plot_embedding
plot_density_parent_tree = imste_single_cell.plot_density_parent_tree
plot_knn_union = imste_visualization.plot_knn_union
interactive_force = imste_visualization.interactive_force


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import ipywidgets as widgets

AVAILABLE_DATASETS = imste_single_cell.available_datasets()
HVG_COUNT_SELECTOR = widgets.Dropdown(
    options=[("500 HVGs", 500), ("1,000 HVGs", 1_000), ("2,000 HVGs", 2_000),
             ("3,000 HVGs", 3_000), ("All genes", None)],
    value=2_000,
    description="HVGs:",
)
PC_COUNT_SELECTOR = widgets.Dropdown(
    options=[("20 PCs", 20), ("30 PCs", 30), ("50 PCs", 50),
             ("75 PCs", 75), ("100 PCs", 100)],
    value=50,
    description="PCs:",
)
MAX_CELLS_SELECTOR = widgets.Dropdown(
    options=[
        ("500 cells", 500),
        ("1,000 cells", 1_000),
        ("2,000 cells", 2_000),
        ("4,000 cells", 4_000),
        ("8,000 cells", 8_000),
        ("All cells", None),
    ],
    value=4_000,
    description="Cell limit:",
)
RANDOM_STATE = 42

EMBEDDING_RENDERER_SELECTOR = widgets.Dropdown(
    options=[("Matplotlib", "matplotlib"), ("Datashader", "datashader")],
    value="datashader",
    description="Renderer:",
)
DATASHADER_BACKGROUND_SELECTOR = widgets.Dropdown(
    options=["white", "black"],
    value="white",
    description="DS background:",
)
DATASHADER_MAX_DIM_SELECTOR = widgets.Dropdown(
    options=[("700 px", 700), ("900 px", 900), ("1,200 px", 1_200), ("1,600 px", 1_600)],
    value=900,
    description="Max plot dimension:",
)

print(f"Datasets: {len(AVAILABLE_DATASETS)}")
display(pd.DataFrame({"Dataset": [path.name for path in AVAILABLE_DATASETS]}))
display(HVG_COUNT_SELECTOR, PC_COUNT_SELECTOR, MAX_CELLS_SELECTOR)
display(
    EMBEDDING_RENDERER_SELECTOR,
    DATASHADER_BACKGROUND_SELECTOR,
    DATASHADER_MAX_DIM_SELECTOR,
)
print(
    "Cell limit: "
    + (f"up to {MAX_CELLS_SELECTOR.value:,}" if MAX_CELLS_SELECTOR.value else "all")
    + " per dataset"
)


Datasets: 12


,Dataset
0,Immune_ALL_hum_mou.h5ad
1,Immune_ALL_human.h5ad
2,Lung_atlas_public.h5ad
3,cellrank-bone-marrow.h5ad
4,cellrank-lung.h5ad
5,cellrank-pancreas.h5ad
6,cellrank-reprogramming-morris.h5ad
7,cellrank-reprogramming-schiebinger.h5ad
8,cellrank-zebrafish.h5ad
9,human_pancreas_norm_complexBatch.h5ad


Dropdown(description='HVGs:', index=2, options=(('500 HVGs', 500), ('1,000 HVGs', 1000), ('2,000 HVGs', 2000),…

Dropdown(description='PCs:', index=2, options=(('20 PCs', 20), ('30 PCs', 30), ('50 PCs', 50), ('75 PCs', 75),…

Dropdown(description='Cell limit:', index=3, options=(('500 cells', 500), ('1,000 cells', 1000), ('2,000 cells…

Dropdown(description='Renderer:', index=1, options=(('Matplotlib', 'matplotlib'), ('Datashader', 'datashader')…

Dropdown(description='DS background:', options=('white', 'black'), value='white')

Dropdown(description='Max plot dimension:', index=1, options=(('700 px', 700), ('900 px', 900), ('1,200 px', 1…

Cell limit: up to 4,000 per dataset


In [12]:
# CLASP preprocessing and graph-construction parameters.
CLASP_PREPROCESS_PARAMS = {
    "n_top_genes": 2_000,  # Updated from HVG_COUNT_SELECTOR before the dataset loop.
    "max_cells": None,
    "min_cell_genes": None,
    "min_gene_counts": 0,
    "normalize": "auto",
    "target_sum": 10_000,
    "log1p": True,
    "scale": True,
}
CLASP_PARAMS = {
    "pca_components": 50,  # Updated from PC_COUNT_SELECTOR before the dataset loop.
    "n_neighbors": 15,
    "intra_fraction": 0.5,
    "n_inter_edges": 1,
    "metric": "euclidean",
    "assignment_quantile": 0.95,
    "hubness_correction": "csls",
    "hubness_k": 10,
    "rank_correction": True,
    "edge_weighting": "distance",
    "inter_edge_mode": "propagate_neighbors",
    "mutual_neighbors": True,
    "neighbor_mode": "distance",
    "symmetrize": True,
}
UMAP_PARAMS = {
    "n_neighbors": 30,
    "min_dist": 0.3,
    "n_epochs": 500,
    "low_memory": True,
    "compatibility_layout": False,
}
IMSTE_PARAMS = {
    "n_msts": 64,
    "n_components": 2,
    "n_epochs": 1000,
    "batch_size": 32_768,
    "learning_rate": 0.05,
    "negative_ratio": 1,
    "lambda_rep": 0.5,
    "random_state": RANDOM_STATE,
    "epsilon": 1e-4,
    "device": "auto",
    "logistic_margin": 1.0,
    "logistic_temperature": 0.5,
    "mst_method": "famst",
    "mst_neighbors": 15,
    "mst_inter_component_edges": 5,
    "mst_max_neighbors": 512,
    "mst_representatives_per_component": 10,
    "pca_init": True,
    "pca_init_scaling": "global",
    "progress_callback": None,
    "mst_progress_callback": None,
}
display(pd.DataFrame({"CLASP": pd.Series(CLASP_PARAMS), "UMAP": pd.Series(UMAP_PARAMS)}))
display(pd.Series(IMSTE_PARAMS, name="IMSTE").to_frame())


,CLASP,UMAP
assignment_quantile,0.95,NaN
compatibility_layout,NaN,False
edge_weighting,distance,NaN
hubness_correction,csls,NaN
hubness_k,10,NaN
inter_edge_mode,propagate_neighbors,NaN
intra_fraction,0.5,NaN
low_memory,NaN,True
metric,euclidean,NaN
min_dist,NaN,0.3


,IMSTE
n_msts,64
n_components,2
n_epochs,1000
batch_size,32768
learning_rate,0.05
negative_ratio,1
lambda_rep,0.5
random_state,42
epsilon,0.0001
device,auto


In [ ]:
from clasp.notebook_workflows import run_clasp_imste_all_datasets

run_clasp_imste_all_datasets(
    AVAILABLE_DATASETS,
    single_cell=imste_single_cell,
    workflow=workflow,
    visualization=imste_visualization,
    imste_embedder_class=IMSTE_EMBEDDER_CLASS,
    clasp_params=CLASP_PARAMS,
    preprocess_params=CLASP_PREPROCESS_PARAMS,
    umap_params=UMAP_PARAMS,
    imste_params=IMSTE_PARAMS,
    hvg_count=HVG_COUNT_SELECTOR.value,
    pc_count=PC_COUNT_SELECTOR.value,
    max_cells=MAX_CELLS_SELECTOR.value,
    random_state=RANDOM_STATE,
    renderer=EMBEDDING_RENDERER_SELECTOR.value,
    datashader_background=DATASHADER_BACKGROUND_SELECTOR.value,
    datashader_max_dim=DATASHADER_MAX_DIM_SELECTOR.value,
)


Preprocessing: 2,000 HVGs, scaled, then 50 PCs; up to 4,000 cells per dataset

[1/12] Immune_ALL_hum_mou.h5ad
Cells: 4,000; genes before HVG selection: 8,135; label: final_annotation; batch: batch
